# Multi-Tape Turing Machine (MTTM) — Supervised Learning
**Theoretical Computer Science Course — INPT**

Simulator of a **5-tape MTTM** executing **batch gradient descent**
for univariate linear regression, using **exact rational arithmetic**.

| Tape | Role | Content |
|------|------|---------|
| `R0` | Dataset | pairs $(x_i, y_i)$ encoded as fractions |
| `R1` | Parameters | $w$ (weight) and $b$ (bias) |
| `R2` | Scratch | accumulators for $\hat{y}$, error, gradients, loss |
| `R3` | Counter | current iteration $t$ |
| `R4` | Output | learned model $f(x) = wx + b$ |

States $Q = \{$ `INIT, LOAD, FORWARD, LOSS, GRAD, UPDATE, CHECK, HALT` $\}$;
transition function $\delta : Q \times \Gamma^5 \to Q \times \Gamma^5 \times \{L,R,S\}^5$
simulated by the `_transition_*` methods.

## 1. Imports and types
Each tape symbol is a `Fraction` (exact) or the blank symbol `B = None`.

In [1]:
import sys
from fractions import Fraction
from dataclasses import dataclass
from typing import Optional
import textwrap

# Avoids UnicodeEncodeError on non-UTF-8 consoles (e.g. Windows cp1252):
# the program prints characters such as δ, η, →, ², █.
try:
    sys.stdout.reconfigure(encoding='utf-8')
except (AttributeError, ValueError):
    pass

# -- Types -------------------------------------------------------------------

Symbol  = Optional[Fraction]   # Gamma: Fraction or B (blank = None)
B: Symbol = None               # blank symbol


## 2. The tape
A bi-infinite tape implemented as a `{position: symbol}` dictionary.

In [2]:
class Tape:
    """
    Tape, potentially infinite in both directions.
    Implemented as a dict {position: Symbol}.
    The read/write head starts at position 0.
    """

    def __init__(self, name: str, initial: list[Symbol] = None):
        self.name = name
        self._cells: dict[int, Symbol] = {}
        self.head: int = 0
        if initial:
            for i, sym in enumerate(initial):
                if sym is not None:
                    self._cells[i] = sym

    def read(self) -> Symbol:
        return self._cells.get(self.head, B)

    def write(self, sym: Symbol):
        if sym is B:
            self._cells.pop(self.head, None)
        else:
            self._cells[self.head] = sym

    def move(self, direction: str):
        """L = left, R = right, S = stay"""
        if direction == 'R':
            self.head += 1
        elif direction == 'L':
            self.head -= 1
        # S -> no movement

    def peek(self, pos: int) -> Symbol:
        return self._cells.get(pos, B)

    def write_at(self, pos: int, sym: Symbol):
        if sym is B:
            self._cells.pop(pos, None)
        else:
            self._cells[pos] = sym

    def snapshot(self) -> str:
        if not self._cells:
            return f"[{self.name}] (empty)"
        positions = sorted(self._cells.keys())
        cells_str = "  ".join(
            f"[{p}:{float(v):.4f}]" if p == self.head else f" {p}:{float(v):.4f} "
            for p, v in ((p, self._cells[p]) for p in positions)
        )
        return f"[{self.name}] head={self.head} | {cells_str}"

    def __repr__(self):
        return self.snapshot()


## 3. Dataset encoding
The $(x, y)$ pairs are laid out on `R0`: `[x0, y0, x1, y1, ...]`, with floats converted to exact fractions.

In [3]:
def encode_dataset(pairs: list[tuple[float, float]]) -> list[Symbol]:
    """
    Encodes (x, y) pairs as symbols on R0.
    Convention: [x0, y0, x1, y1, ..., xn-1, yn-1]
    Floats are converted to Fraction (exact representation).
    """
    symbols = []
    for x, y in pairs:
        symbols.append(Fraction(x).limit_denominator(10**6))
        symbols.append(Fraction(y).limit_denominator(10**6))
    return symbols


## 4. Hyperparameters
`limit_denom=None` => **exact** arithmetic (default). Setting it (e.g. `10**6`) bounds denominator growth and speeds up large `T`.

In [4]:
@dataclass
class MTMConfig:
    """MTTM hyperparameters."""
    eta: Fraction = Fraction(1, 100)   # learning rate eta
    T: int = 100                        # maximum number of iterations
    verbose: bool = True
    log_every: int = 10                 # print every N iterations
    # Optional: bounds the denominator of w and b after each update.
    # None  -> EXACT rational arithmetic (default, faithful to the TM model,
    #          but denominators grow -> slows down for large T).
    # 10**6 -> near-exact (6+ decimals) and MUCH faster for large T.
    limit_denom: Optional[int] = None


## 5. The machine — transition function $\delta$
The core of the simulator: one method per state, plus `run()`, `predict()` and `evaluate()`.

In [5]:
class MultiTapeTM:
    """
    5-tape Multi-Tape Turing Machine simulator.

    Tapes (indices):
        0 - Dataset    R0
        1 - Parameters R1  (positions 0=w, 1=b)
        2 - Scratch    R2  (positions 0=grad_w, 1=grad_b, 2=loss_accum)
        3 - Counter    R3  (position 0=t)
        4 - Output     R4  (positions 0=w_final, 1=b_final)
    """

    STATES = ['INIT', 'LOAD', 'FORWARD', 'LOSS', 'GRAD', 'UPDATE', 'CHECK', 'HALT']

    def __init__(self, dataset: list[tuple[float, float]], cfg: MTMConfig = None):
        self.cfg = cfg or MTMConfig()
        self.n = len(dataset)

        # Tape initialization
        self.tapes = [
            Tape("R0-Dataset",    encode_dataset(dataset)),
            Tape("R1-Params",     [Fraction(0), Fraction(0)]),  # w=0, b=0
            Tape("R2-Scratch",    []),
            Tape("R3-Counter",    [Fraction(0)]),
            Tape("R4-Output",     []),
        ]

        self.state: str = 'INIT'
        self.step_count: int = 0
        self.history: list[dict] = []   # transition trace

        # Internal registers (control head working memory)
        # In a formal MTTM these values would be encoded on R2.
        # Here they are exposed explicitly for pedagogical clarity.
        self._sum_grad_w = Fraction(0)
        self._sum_grad_b = Fraction(0)
        self._loss       = Fraction(0)
        self._data_ptr   = 0   # current pointer into R0

    # -- Tape reads ------------------------------------------------------

    @property
    def w(self) -> Fraction:
        v = self.tapes[1].peek(0)
        return v if v is not None else Fraction(0)

    @property
    def b(self) -> Fraction:
        v = self.tapes[1].peek(1)
        return v if v is not None else Fraction(0)

    @property
    def t(self) -> int:
        v = self.tapes[3].peek(0)
        return int(v) if v is not None else 0

    # -- Transitions delta -------------------------------------------------

    def _transition_INIT(self):
        """
        delta(INIT, ...) -> LOAD
        Initializes w=0, b=0 on R1, t=0 on R3.
        """
        self.tapes[1].write_at(0, Fraction(0))
        self.tapes[1].write_at(1, Fraction(0))
        self.tapes[3].write_at(0, Fraction(0))
        self._data_ptr = 0
        self._log_transition('INIT', 'LOAD', 'Initialization w=0, b=0, t=0')
        self.state = 'LOAD'

    def _transition_LOAD(self):
        """
        delta(LOAD, ...) -> FORWARD
        Resets the data pointer to 0 for a new full pass.
        Resets the gradient accumulators on R2 to zero.
        """
        self._data_ptr   = 0
        self._sum_grad_w = Fraction(0)
        self._sum_grad_b = Fraction(0)
        self._loss       = Fraction(0)
        self.tapes[2].write_at(0, Fraction(0))
        self.tapes[2].write_at(1, Fraction(0))
        self.tapes[2].write_at(2, Fraction(0))
        self._log_transition('LOAD', 'FORWARD', f'Starting epoch t={self.t}, accumulators reset to 0')
        self.state = 'FORWARD'

    def _transition_FORWARD(self):
        """
        delta(FORWARD, ...) -> LOSS (after computing y_hat for every xi)
        Reads (x_i, y_i) on R0, reads w,b on R1, writes y_hat_i on R2.
        Loops over every example.
        """
        w, b = self.w, self.b
        pairs = []
        for i in range(self.n):
            x = self.tapes[0].peek(2 * i)
            y = self.tapes[0].peek(2 * i + 1)
            y_hat = w * x + b                  # exact linear computation (Fraction)
            pairs.append((x, y, y_hat))

        # Stores the results in the R2 working register
        self._pairs = pairs
        self._log_transition('FORWARD', 'LOSS',
            f'y_hat computed for {self.n} examples (w={float(w):.4f}, b={float(b):.4f})')
        self.state = 'LOSS'

    def _transition_LOSS(self):
        """
        delta(LOSS, ...) -> GRAD
        Computes L = (1/n) sum (y_hat_i - y_i)^2 and writes it to R2[2].
        """
        loss = Fraction(0)
        for (x, y, y_hat) in self._pairs:
            loss += (y_hat - y) ** 2
        loss = loss / self.n
        self._loss = loss
        self.tapes[2].write_at(2, loss)
        self._log_transition('LOSS', 'GRAD', f'L = {float(loss):.6f}')
        self.state = 'GRAD'

    def _transition_GRAD(self):
        """
        delta(GRAD, ...) -> UPDATE
        Computes grad_w L = (2/n) sum (y_hat_i - y_i)*x_i
                 grad_b L = (2/n) sum (y_hat_i - y_i)
        Writes the gradients to R2[0] and R2[1].
        """
        grad_w = Fraction(0)
        grad_b = Fraction(0)
        coef   = Fraction(2, self.n)
        for (x, y, y_hat) in self._pairs:
            err = y_hat - y
            grad_w += coef * err * x
            grad_b += coef * err
        self._sum_grad_w = grad_w
        self._sum_grad_b = grad_b
        self.tapes[2].write_at(0, grad_w)
        self.tapes[2].write_at(1, grad_b)
        self._log_transition('GRAD', 'UPDATE',
            f'grad_w={float(grad_w):.6f}, grad_b={float(grad_b):.6f}')
        self.state = 'UPDATE'

    def _transition_UPDATE(self):
        """
        delta(UPDATE, ...) -> CHECK
        Reads R2[0]=grad_w, R2[1]=grad_b and eta=cfg.eta.
        Writes w' = w - eta*grad_w and b' = b - eta*grad_b to R1.
        """
        eta   = self.cfg.eta
        new_w = self.w - eta * self._sum_grad_w
        new_b = self.b - eta * self._sum_grad_b
        # Optional denominator bound (see MTMConfig.limit_denom).
        if self.cfg.limit_denom is not None:
            new_w = new_w.limit_denominator(self.cfg.limit_denom)
            new_b = new_b.limit_denominator(self.cfg.limit_denom)
        self.tapes[1].write_at(0, new_w)
        self.tapes[1].write_at(1, new_b)
        self._log_transition('UPDATE', 'CHECK',
            f'w: {float(self.w + eta*self._sum_grad_w):.4f} -> {float(new_w):.4f} | '
            f'b: {float(self.b + eta*self._sum_grad_b):.4f} -> {float(new_b):.4f}')
        self.state = 'CHECK'

    def _transition_CHECK(self):
        """
        delta(CHECK, t, ...) ->
            LOAD  if t+1 < T
            HALT  if t+1 >= T
        Increments the counter on R3.
        """
        t_new = self.t + 1
        self.tapes[3].write_at(0, Fraction(t_new))
        if t_new >= self.cfg.T:
            # Writes the final model to R4
            self.tapes[4].write_at(0, self.w)
            self.tapes[4].write_at(1, self.b)
            self._log_transition('CHECK', 'HALT',
                f't={t_new} >= T={self.cfg.T} -> q_acc (accept) | '
                f'Model: w={float(self.w):.6f}, b={float(self.b):.6f}')
            self.state = 'HALT'
        else:
            self._log_transition('CHECK', 'LOAD', f't={t_new} < T={self.cfg.T} -> new epoch')
            self.state = 'LOAD'

    # -- Main loop ---------------------------------------------------------

    def _log_transition(self, from_state: str, to_state: str, info: str):
        entry = {
            'step':  self.step_count,
            'from':  from_state,
            'to':    to_state,
            'info':  info,
            'w':     float(self.w),
            'b':     float(self.b),
            'loss':  float(self._loss),
            't':     self.t,
        }
        self.history.append(entry)

        if self.cfg.verbose:
            should_print = (
                from_state in ('INIT', 'HALT') or
                (from_state == 'CHECK' and self.t % self.cfg.log_every == 0)
            )
            if should_print:
                print(f"  delta({from_state:8s} -> {to_state:8s}) | step={self.step_count:4d} | {info}")

    def run(self) -> dict:
        """
        Runs the MTTM until the HALT state.
        Returns a summary of the results.
        """
        print("=" * 70)
        print("  5-Tape MTTM -- Supervised Learning (Linear Regression)")
        print(f"  n={self.n} examples | eta={float(self.cfg.eta):.4f} | T={self.cfg.T} iterations")
        print("=" * 70)

        _transitions = {
            'INIT':    self._transition_INIT,
            'LOAD':    self._transition_LOAD,
            'FORWARD': self._transition_FORWARD,
            'LOSS':    self._transition_LOSS,
            'GRAD':    self._transition_GRAD,
            'UPDATE':  self._transition_UPDATE,
            'CHECK':   self._transition_CHECK,
        }

        while self.state != 'HALT':
            _transitions[self.state]()
            self.step_count += 1

        print("=" * 70)
        print(f"  HALT reached in {self.step_count} transitions.")
        print(f"  Final model (R4): w = {float(self.w):.6f}, b = {float(self.b):.6f}")
        print("=" * 70)

        return {
            'w':           float(self.w),
            'b':           float(self.b),
            'loss_final':  float(self._loss),
            'steps':       self.step_count,
            'history':     self.history,
        }

    def predict(self, x: float) -> float:
        return float(self.w) * x + float(self.b)

    def evaluate(self, test_pairs: list[tuple[float, float]]) -> dict:
        """Computes MSE and R2 on a test set."""
        y_true  = [y for _, y in test_pairs]
        y_pred  = [self.predict(x) for x, _ in test_pairs]
        n       = len(y_true)
        mse     = sum((p - t) ** 2 for p, t in zip(y_pred, y_true)) / n
        mean_y  = sum(y_true) / n
        ss_tot  = sum((t - mean_y) ** 2 for t in y_true)
        ss_res  = sum((p - t) ** 2 for p, t in zip(y_pred, y_true))
        r2      = 1 - ss_res / ss_tot if ss_tot != 0 else 0
        return {'MSE': round(mse, 6), 'R2': round(r2, 6)}


## 6. Utility — tape display

In [6]:
def print_tapes(tm: MultiTapeTM):
    print("\n-- Tape state --------------------------------------------------")
    for tape in tm.tapes:
        print(" ", tape.snapshot())
    print()


## 7. Demo 1 — $y = 2x + 1$
The MTTM should recover $w \approx 2$, $b \approx 1$.

In [7]:
def demo_simple():
    """
    Example 1: y = 2x + 1 with simulated Gaussian noise.
    The MTTM should recover w~=2, b~=1.
    """
    print("\n" + "#" * 70)
    print("  DEMO 1 -- Regression y = 2x + 1 (10 points)")
    print("#" * 70)

    # Dataset built by hand (no numpy import, to stay faithful to the TM spirit)
    dataset = [
        (1.0,  3.1), (2.0,  5.0), (3.0,  6.9), (4.0,  9.1),
        (5.0, 10.8), (6.0, 13.2), (7.0, 15.0), (8.0, 17.1),
        (9.0, 18.9), (10.0, 21.2),
    ]

    cfg = MTMConfig(eta=Fraction(1, 100), T=500, verbose=True, log_every=100)
    tm  = MultiTapeTM(dataset, cfg)
    res = tm.run()

    print(f"\n  Theoretical values: w=2.0000, b=1.0000")
    print(f"  Learned values    : w={res['w']:.6f}, b={res['b']:.6f}")
    print(f"  Total transitions : {res['steps']}")

    eval_res = tm.evaluate(dataset)
    print(f"  MSE={eval_res['MSE']:.6f} | R2={eval_res['R2']:.6f}")
    return tm, res


tm1, r1 = demo_simple()


######################################################################
  DEMO 1 -- Regression y = 2x + 1 (10 points)
######################################################################
  5-Tape MTTM -- Supervised Learning (Linear Regression)
  n=10 examples | eta=0.0100 | T=500 iterations
  delta(INIT     -> LOAD    ) | step=   0 | Initialization w=0, b=0, t=0
  delta(CHECK    -> LOAD    ) | step= 600 | t=100 < T=500 -> new epoch
  delta(CHECK    -> LOAD    ) | step=1200 | t=200 < T=500 -> new epoch
  delta(CHECK    -> LOAD    ) | step=1800 | t=300 < T=500 -> new epoch


  delta(CHECK    -> LOAD    ) | step=2400 | t=400 < T=500 -> new epoch


  delta(CHECK    -> HALT    ) | step=3000 | t=500 >= T=500 -> q_acc (accept) | Model: w=2.019862, b=0.903240
  HALT reached in 3001 transitions.
  Final model (R4): w = 2.019862, b = 0.903240

  Theoretical values: w=2.0000, b=1.0000
  Learned values    : w=2.019862, b=0.903240
  Total transitions : 3001
  MSE=0.017079 | R2=0.999487


## 8. Demo 2 — $y = -0.5x + 10$
Negative relationship, more cautious learning rate.

In [8]:
def demo_harder():
    """
    Example 2: y = -0.5x + 10 (negative relationship).
    """
    print("\n" + "#" * 70)
    print("  DEMO 2 -- Regression y = -0.5x + 10 (8 points)")
    print("#" * 70)

    dataset = [
        (0.0, 10.2), (2.0,  9.1), (4.0,  8.0), (6.0,  7.1),
        (8.0,  6.0), (10.0, 5.2), (12.0, 4.1), (14.0, 2.9),
    ]

    cfg = MTMConfig(eta=Fraction(1, 200), T=800, verbose=True, log_every=200)
    tm  = MultiTapeTM(dataset, cfg)
    res = tm.run()

    print(f"\n  Theoretical values: w=-0.5000, b=10.0000")
    print(f"  Learned values    : w={res['w']:.6f}, b={res['b']:.6f}")
    eval_res = tm.evaluate(dataset)
    print(f"  MSE={eval_res['MSE']:.6f} | R2={eval_res['R2']:.6f}")
    return tm, res


tm2, r2 = demo_harder()


######################################################################
  DEMO 2 -- Regression y = -0.5x + 10 (8 points)
######################################################################
  5-Tape MTTM -- Supervised Learning (Linear Regression)
  n=8 examples | eta=0.0050 | T=800 iterations
  delta(INIT     -> LOAD    ) | step=   0 | Initialization w=0, b=0, t=0
  delta(CHECK    -> LOAD    ) | step=1200 | t=200 < T=800 -> new epoch


  delta(CHECK    -> LOAD    ) | step=2400 | t=400 < T=800 -> new epoch


  delta(CHECK    -> LOAD    ) | step=3600 | t=600 < T=800 -> new epoch


  delta(CHECK    -> HALT    ) | step=4800 | t=800 >= T=800 -> q_acc (accept) | Model: w=-0.415702, b=9.207428
  HALT reached in 4801 transitions.
  Final model (R4): w = -0.415702, b = 9.207428

  Theoretical values: w=-0.5000, b=10.0000
  Learned values    : w=-0.415702, b=9.207428
  MSE=0.269322 | R2=0.950668


## 9. Demo 3 — full formal trace
Every transition $\delta$ is printed (3 examples, 3 iterations).

In [9]:
def demo_formal_trace():
    """
    Example 3 -- Full formal trace over 3 examples, 3 iterations.
    Shows every delta transition for pedagogical purposes.
    """
    print("\n" + "#" * 70)
    print("  DEMO 3 -- Formal trace (3 examples, 3 iterations)")
    print("#" * 70)

    dataset = [(1.0, 2.0), (2.0, 4.0), (3.0, 6.0)]   # y = 2x exactly
    cfg = MTMConfig(eta=Fraction(1, 50), T=3, verbose=True, log_every=1)
    tm  = MultiTapeTM(dataset, cfg)
    res = tm.run()

    print("\n-- Full transition trace ----------------------------------------")
    for h in res['history']:
        print(f"  step {h['step']:3d} | delta({h['from']:8s}->{h['to']:8s}) "
              f"| t={h['t']:3d} | L={h['loss']:.6f} | w={h['w']:.4f}, b={h['b']:.4f}")

    print_tapes(tm)
    return tm, res


tm3, r3 = demo_formal_trace()


######################################################################
  DEMO 3 -- Formal trace (3 examples, 3 iterations)
######################################################################
  5-Tape MTTM -- Supervised Learning (Linear Regression)
  n=3 examples | eta=0.0200 | T=3 iterations
  delta(INIT     -> LOAD    ) | step=   0 | Initialization w=0, b=0, t=0
  delta(CHECK    -> LOAD    ) | step=   6 | t=1 < T=3 -> new epoch
  delta(CHECK    -> LOAD    ) | step=  12 | t=2 < T=3 -> new epoch
  delta(CHECK    -> HALT    ) | step=  18 | t=3 >= T=3 -> q_acc (accept) | Model: w=0.890833, b=0.379250
  HALT reached in 19 transitions.
  Final model (R4): w = 0.890833, b = 0.379250

-- Full transition trace ----------------------------------------
  step   0 | delta(INIT    ->LOAD    ) | t=  0 | L=0.000000 | w=0.0000, b=0.0000
  step   1 | delta(LOAD    ->FORWARD ) | t=  0 | L=0.000000 | w=0.0000, b=0.0000
  step   2 | delta(FORWARD ->LOSS    ) | t=  0 | L=0.000000 | w=0.0000, b=0.0000


## 10. Complexity analysis

In [10]:
def show_complexity():
    print("\n" + "-" * 70)
    print("  FORMAL COMPLEXITY ANALYSIS")
    print("-" * 70)
    msg = textwrap.dedent("""
    Let:
      n = number of examples in the dataset
      T = maximum number of iterations (epochs)
      k = number of bits used to encode a symbol (fixed-point)

    Transitions per iteration:
      LOAD     : O(1)          -- resets the accumulators
      FORWARD  : O(n*k)        -- reads R0 + multiplication on k bits
      LOSS     : O(n*k)        -- accumulates (y_hat-y)^2
      GRAD     : O(n*k)        -- accumulates the gradients
      UPDATE   : O(k)          -- writes w', b' to R1
      CHECK    : O(k)          -- increments and compares the counter

    Total complexity: O(T * n * k) transitions

    Reduction to a single-tape TM:
      Quadratic overhead O((T*n*k)^2) under standard simulation
      -> the algorithm remains in PTIME (Church-Turing computable)

    Space:
      R0 : O(n*k)   -- fixed dataset
      R1 : O(k)     -- two parameters
      R2 : O(k)     -- bounded accumulators
      R3 : O(log T) -- binary counter
      R4 : O(k)     -- final output
      Total : O(n*k) -- polynomial space
    """)
    print(msg)


show_complexity()


----------------------------------------------------------------------
  FORMAL COMPLEXITY ANALYSIS
----------------------------------------------------------------------

Let:
  n = number of examples in the dataset
  T = maximum number of iterations (epochs)
  k = number of bits used to encode a symbol (fixed-point)

Transitions per iteration:
  LOAD     : O(1)          -- resets the accumulators
  FORWARD  : O(n*k)        -- reads R0 + multiplication on k bits
  LOSS     : O(n*k)        -- accumulates (y_hat-y)^2
  GRAD     : O(n*k)        -- accumulates the gradients
  UPDATE   : O(k)          -- writes w', b' to R1
  CHECK    : O(k)          -- increments and compares the counter

Total complexity: O(T * n * k) transitions

Reduction to a single-tape TM:
  Quadratic overhead O((T*n*k)^2) under standard simulation
  -> the algorithm remains in PTIME (Church-Turing computable)

Space:
  R0 : O(n*k)   -- fixed dataset
  R1 : O(k)     -- two parameters
  R2 : O(k)     -- bounded accu